In [1]:
import pandas as pd
ct = pd.read_excel('output/Control-Totals-LUVit.xlsx', sheet_name='HH')
gen = pd.read_excel('output/LUVit_ct_by_tod_generator-2026-04-30_90-90-90.xlsx', sheet_name='HH')

ct_ids  = set(ct['control_id'])
gen_ids = set(gen['subreg_id'].unique())  # subreg, but check nosplit too via HHwork sheet
hhwork  = pd.read_excel('output/LUVit_ct_by_tod_generator-2026-04-30_90-90-90.xlsx', sheet_name='HHwork')
missing = ct_ids - set(hhwork['nosplit_geo_id'])
print('missing control_ids:', sorted(missing))
print('HH 2060 dropped:', ct.loc[ct['control_id'].isin(missing), '2060'].sum())

# capacity-binding geos
cap_bound = hhwork.groupby('nosplit_geo_id').apply(
    lambda g: g['DUnetcapacity'].sum() < (g['target_growth_final'].sum() - g['HH2023'].sum())
)
print('capacity-bound nosplit geos:', cap_bound[cap_bound].index.tolist())

missing control_ids: []
HH 2060 dropped: 0.0
capacity-bound nosplit geos: [49, 57, 168, 401, 404]


In [2]:
hhwork

,subreg_id,nosplit_geo_id,RGID,name,is_tod,DUtotcapacity,DUnetcapacity,DUcapshare,target_growth_ini,target_growth_final,HH2023,PPH2023,HH2060,PPH2060,target_share
0,1001,1,1,Bellevue,1,81534,51084,76.1,36124,42726,30450,2.162167,73176,2.180606,90.0
1,1,1,1,Bellevue,0,47605,16048,23.9,11349,4747,31557,2.739170,36304,2.724936,10.0
2,1002,2,1,Seattle,1,575123,293589,89.4,62065,62494,281534,1.911602,344028,2.136478,90.0
3,2,2,1,Seattle,0,123688,34878,10.6,7373,6944,88810,2.363641,95754,2.456920,10.0
4,1003,3,2,Auburn,1,17526,9190,30.0,4609,5106,8336,2.760677,13442,2.573972,33.2
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
226,402,402,6,NBK - Bangor,0,1012,0,100.0,465,465,1012,3.469368,1477,3.226054,100.0
227,1403,403,6,JBLM - Rural,1,0,0,0.0,135,135,0,0.000000,135,1.840773,93.8
228,403,403,6,JBLM - Rural,0,1089,9,100.0,9,9,1080,3.675000,1089,3.663082,6.2
229,404,404,1,Naval Station Everett,0,139,139,100.0,235,235,0,0.000000,235,1.726713,100.0


# Trace control_id 157 through the split pipeline

control_id 157 appears in `Control-Totals-LUVit.xlsx` but not in the split output. The split step drops control_ids that aren't represented in the parcel base-data feeding the HCT split. Below we walk the data sources to find where 157 disappears.

In [3]:
import sys
from pathlib import Path

# Make the project importable so we can use Pipeline
project_root = Path.cwd()
while project_root.name and not (project_root / 'control_totals' / 'util').exists():
    project_root = project_root.parent
sys.path.insert(0, str(project_root / 'control_totals'))

from util import Pipeline

configs_dir = Path.cwd() / 'configs'
p = Pipeline(settings_path=str(configs_dir))
print('pipeline.h5:', p.get_pipeline_path() if hasattr(p, 'get_pipeline_path') else '(see Pipeline)')
CID = 157

pipeline.h5: (see Pipeline)


## 1. Is 157 in the Control-Totals workbook?

In [4]:
for sheet in ['HH', 'HHPop', 'Emp']:
    df = pd.read_excel('output/Control-Totals-LUVit.xlsx', sheet_name=sheet)
    row = df.loc[df['control_id'] == CID]
    print(f'--- {sheet} ---')
    print(row.to_string(index=False) if not row.empty else 'NOT PRESENT')

--- HH ---
 control_id      2023      2025      2030      2035      2040      2044     2050       2055       2060
        157 34.930001 39.095947 49.510811 59.925676 70.340541 70.406487 91.17027 101.585135 106.212309
--- HHPop ---
 control_id    2023       2025       2030       2035       2040       2044       2050       2055       2060
        157 102.114 112.702432 139.173513 165.644594 192.115676 164.074379 245.057838 271.528919 246.003078
--- Emp ---
 control_id  2023     2025      2030      2035      2040       2044      2050       2055      2060
        157     0 7.243243 25.351351 43.459459 61.567568 106.159703 97.783784 115.891892 155.06827


## 2. Does 157 exist in the parcel→control xwalk and the target xwalk?

`current_parcel_control_area_xwalk` is the bridge between MySQL parcel base data and control geographies. If 157 isn't here, no parcels are tagged to it and the split step will never see it.

In [5]:
parcels_xwalk = p.get_table('current_parcel_control_area_xwalk')
print('Total parcels in xwalk:', len(parcels_xwalk))
print(f'Parcels tagged to control_id={CID}:',
      (parcels_xwalk['control_id'] == CID).sum())
print('Distinct subreg_ids for 157:',
      parcels_xwalk.loc[parcels_xwalk['control_id'] == CID, 'subreg_id'].unique())

ct_xwalk = p.get_table('control_target_xwalk')
print('\ncontrol_target_xwalk row for 157:')
print(ct_xwalk.loc[ct_xwalk['control_id'] == CID].to_string(index=False))

Total parcels in xwalk: 1329944
Parcels tagged to control_id=157: 67
Distinct subreg_ids for 157: [157]

control_target_xwalk row for 157:
 target_id    target_name  control_id   control_name  county_id  exclude_from_target  rgid
       157 Darrington UGA         157 Darrington UGA      53061                    0     5


## 3. Is 157 in the cached split-HCT base data?

The split step reads `split_hct_base_data_<base_year>` from `pipeline.h5`. If 157 isn't here, it's dropped before any merging with the control sheets — `aggregate_base_data` inner-joins parcels to the xwalk and groups by `(subreg_id, control_id)`, so any control_id without parcels (or with only parcels that have 0 households / 0 jobs and weren't selected) won't appear.

In [6]:
base_year = p.settings['base_year']
table_name = f'split_hct_base_data_{base_year}'
base_data = p.get_table(table_name)
print('base_data columns:', list(base_data.columns))
print(f'\nbase_data rows for nosplit_geo_id={CID}:')
print(base_data.loc[base_data['nosplit_geo_id'] == CID].to_string(index=False))
print('\nIs 157 in base_data?', CID in set(base_data['nosplit_geo_id']))

base_data columns: ['split_geo_id', 'nosplit_geo_id', 'households', 'persons', 'jobs', 'name', 'RGID']

base_data rows for nosplit_geo_id=157:
 split_geo_id  nosplit_geo_id  households  persons  jobs           name  RGID
          157             157        31.0     78.0  14.0 Darrington UGA     5

Is 157 in base_data? True


## 4. What does control_id 157 represent?

Look it up in the Elmer control-areas table (loaded by Elmer settings) so we know which jurisdiction/area we're chasing.

In [7]:
for tbl in ['control_areas', 'old_control_areas']:
    try:
        ca = p.get_table(tbl)
    except Exception as e:
        print(f'{tbl}: not available ({e})')
        continue
    row = ca.loc[ca.index == CID] if ca.index.name and ca.index.name.startswith('control') else ca.loc[ca.get('control_id', pd.Series(dtype=int)) == CID]
    print(f'--- {tbl} ---')
    print(row.to_string() if not row.empty else 'NOT PRESENT')

--- control_areas ---
     control_id    control_name                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                   

## 5. Parcel vintage mismatch?

`aggregate_base_data` joins MySQL base data (parcel_base_year, e.g. 2018) to `current_parcel_control_area_xwalk` (built from 2023 parcels). If parcel_ids changed between vintages, control_ids whose 2018 parcels aren't in the 2023 xwalk will be dropped.

Check whether a non-`current` xwalk exists with 2018-vintage parcel_ids, and whether 157 has parcels there.

In [8]:
for tbl in ['parcel_control_area_xwalk', 'current_parcel_control_area_xwalk']:
    try:
        x = p.get_table(tbl)
    except Exception as e:
        print(f'{tbl}: not available ({e})')
        continue
    n_total = len(x)
    n157 = (x['control_id'] == CID).sum()
    print(f'{tbl}: total={n_total:,}  parcels for 157={n157}')

parcel_control_area_xwalk: not available ('No object named parcel_control_area_xwalk in the file')
current_parcel_control_area_xwalk: total=1,329,944  parcels for 157=67


## 6. How many control_ids in the controls workbook are missing from base_data?

If this is a systematic issue, lots of small/UGA control_ids will be missing — confirming the inner-join in `aggregate_base_data` (parcel_base_year MySQL ⋈ current parcel xwalk) is the gate.

In [9]:
ct_hh = pd.read_excel('output/Control-Totals-LUVit.xlsx', sheet_name='HH')
in_workbook = set(ct_hh['control_id'])
in_basedata = set(base_data['nosplit_geo_id'])
in_parcel_xwalk = set(parcels_xwalk['control_id'])

missing_basedata = sorted(in_workbook - in_basedata)
missing_xwalk = sorted(in_workbook - in_parcel_xwalk)

print(f'In workbook : {len(in_workbook)}')
print(f'In base_data: {len(in_basedata)}')
print(f'In parcel xwalk: {len(in_parcel_xwalk)}')
print(f'\nMissing from base_data ({len(missing_basedata)}):')
print(missing_basedata)
print(f'\nMissing from parcel xwalk ({len(missing_xwalk)}):')
print(missing_xwalk)
print('\nIn xwalk but missing from base_data (=> have parcels but none in MySQL hh/jobs):')
have_parcels_no_basedata = sorted((in_workbook & in_parcel_xwalk) - in_basedata)
print(have_parcels_no_basedata)

In workbook : 156
In base_data: 156
In parcel xwalk: 156

Missing from base_data (0):
[]

Missing from parcel xwalk (0):
[]

In xwalk but missing from base_data (=> have parcels but none in MySQL hh/jobs):
[]


## 7. Quantify the 2060 HH shortfall

Compare per-control_id 2060 HH targets in `Control-Totals-LUVit.xlsx` against what came out of the split. Decompose the shortfall into:

- **Missing**: control_id not in split output at all
- **Capacity-bound**: split-aggregated `HHbase + DUnetcapacity` < workbook target (no room to fit it)
- **Other**: split has the geo and capacity is sufficient, yet output < target (algorithmic, e.g. minshare floor, PPH constraints, or growth>0 minimum)

In [10]:
GEN = 'output/LUVit_ct_by_tod_generator-2026-04-30_90-90-90.xlsx'
hhwork = pd.read_excel(GEN, sheet_name='HHwork')
ct_hh  = pd.read_excel('output/Control-Totals-LUVit.xlsx', sheet_name='HH')

# Year columns may be int or str
def col(df, year):
    return year if year in df.columns else str(year)

base_year = 2023
end_year = 2060
trg = ct_hh.set_index('control_id')[col(ct_hh, end_year)].rename('trg')

split_agg = hhwork.groupby('nosplit_geo_id').agg(
    HHbase=(f'HH{base_year}', 'sum'),
    HHtarget=(f'HH{end_year}', 'sum'),
    DUnetcapacity=('DUnetcapacity', 'sum'),
).rename_axis('control_id')

cmp = split_agg.join(trg, how='outer')
cmp['gap'] = cmp['trg'] - cmp['HHtarget']
cmp['max_possible'] = cmp['HHbase'] + cmp['DUnetcapacity']
cmp['cap_short'] = (cmp['trg'] - cmp['max_possible']).clip(lower=0)

print(f'Workbook total {end_year} HH    :', round(cmp['trg'].sum()))
print(f'Split    total {end_year} HH    :', round(cmp['HHtarget'].sum()))
print(f'Total gap                       :', round(cmp['gap'].sum()))
print()
print('# control_ids missing from split:', cmp['HHtarget'].isna().sum())
print('Missing-target HH                :', round(cmp.loc[cmp['HHtarget'].isna(), 'trg'].sum()))
print()
cap_bound = cmp[cmp['cap_short'] > 0]
print(f'# capacity-bound control areas  : {len(cap_bound)}')
print('Sum of pure capacity shortfall   :', round(cap_bound['cap_short'].sum()))
print()
positive_gap = cmp.loc[cmp['HHtarget'].notna() & (cmp['gap'] > 0), 'gap'].sum()
print('Total positive gap (non-missing) :', round(positive_gap))
print('Other (non-capacity) shortfall   :', round(positive_gap - cap_bound['cap_short'].sum()))

Workbook total 2060 HH    : 2449926
Split    total 2060 HH    : 2462921
Total gap                       : -12995

# control_ids missing from split: 0
Missing-target HH                : 0

# capacity-bound control areas  : 16
Sum of pure capacity shortfall   : 10946

Total positive gap (non-missing) : 3039
Other (non-capacity) shortfall   : -7907


## 8. Where is the gap?

Capacity-bound only accounts for ~10k of the 117k shortfall. There's also a 285k positive gap and a ~168k overshoot — they net to 117k. Look at where in particular: largest under- and over-shoots, and breakdown by RGID.

In [11]:
ct_xwalk = p.get_table('control_target_xwalk')[['control_id', 'control_name', 'county_id', 'rgid']].drop_duplicates()
cmp2 = cmp.reset_index().merge(ct_xwalk, on='control_id', how='left')

print('--- By RGID (rgid 1=metro, 2=core, 3=HCT, 4=cities, 5=UGA, 6=rural) ---')
print(cmp2.groupby('rgid').agg(
    n=('control_id', 'size'),
    workbook=('trg', 'sum'),
    split=('HHtarget', 'sum'),
    gap=('gap', 'sum'),
    cap_short=('cap_short', 'sum'),
).round().to_string())

print('\n--- By county ---')
print(cmp2.groupby('county_id').agg(
    workbook=('trg', 'sum'),
    split=('HHtarget', 'sum'),
    gap=('gap', 'sum'),
).round().to_string())

print(f'\nUndershoots (gap > 0): {(cmp2["gap"] > 0).sum()} controls, sum={round(cmp2.loc[cmp2["gap"]>0,"gap"].sum())}')
print(f'Overshoots  (gap < 0): {(cmp2["gap"] < 0).sum()} controls, sum={round(cmp2.loc[cmp2["gap"]<0,"gap"].sum())}')

--- By RGID (rgid 1=metro, 2=core, 3=HCT, 4=cities, 5=UGA, 6=rural) ---
       n  workbook   split     gap  cap_short
rgid                                         
1      8  829260.0  831708 -2448.0      507.0
2     18  590843.0  593498 -2655.0        0.0
3     36  561393.0  562460 -1067.0     9277.0
4     50  175663.0  176157  -494.0      240.0
5     33   80236.0   84314 -4078.0      796.0
6     11  212531.0  214784 -2253.0      125.0

--- By county ---
            workbook    split     gap
county_id                            
53033      1264732.0  1269201 -4469.0
53035       146426.0   148519 -2093.0
53053       489680.0   494685 -5005.0
53061       549089.0   550516 -1427.0

Undershoots (gap > 0): 61 controls, sum=3039
Overshoots  (gap < 0): 94 controls, sum=-16034


## 9. Hypothesis: base-year mismatch

The split uses `parcel_base_year=2018` MySQL household counts as `HHbase`. But `Control-Totals-LUVit.xlsx` has 2023 as the base year. The split computes:

```
HH2060 = HHbase_from_MySQL_2018 + (workbook_2060 - workbook_2023)
```

If `MySQL_2018_HH` differs from `workbook_2023_HH` for a control_id, the final 2060 is off by exactly that delta. Fast-growing metros: 2018 < 2023 → undershoot. Slow / declining places: 2018 > 2023 → overshoot.

In [12]:
# Compare MySQL 2018 base vs workbook 2023 base per control_id
mysql_base = base_data.groupby('nosplit_geo_id')['households'].sum().rename('mysql_2018_hh').rename_axis('control_id')
wb_base    = ct_hh.set_index('control_id')[col(ct_hh, base_year)].rename('workbook_2023_hh')

base_cmp = pd.concat([mysql_base, wb_base], axis=1)
base_cmp['delta_2018_to_2023'] = base_cmp['workbook_2023_hh'] - base_cmp['mysql_2018_hh']
base_cmp = base_cmp.merge(ct_xwalk[['control_id', 'rgid']], left_index=True, right_on='control_id', how='left')

print('Total MySQL 2018 HH    :', round(base_cmp['mysql_2018_hh'].sum()))
print('Total workbook 2023 HH :', round(base_cmp['workbook_2023_hh'].sum()))
print('Implied 2018->2023 grow:', round(base_cmp['delta_2018_to_2023'].sum()))
print()
print('Delta (workbook2023 - mysql2018) by RGID:')
print(base_cmp.groupby('rgid')['delta_2018_to_2023'].sum().round().to_string())
print()
print('Compare to HH gap (workbook2060 - split2060) by RGID:')
print(cmp2.groupby('rgid')['gap'].sum().round().to_string())

Total MySQL 2018 HH    : 1736132
Total workbook 2023 HH : 1723132
Implied 2018->2023 grow: -13000

Delta (workbook2023 - mysql2018) by RGID:
rgid
1   -2450.0
2   -2656.0
3   -1069.0
4    -495.0
5   -4075.0
6   -2254.0

Compare to HH gap (workbook2060 - split2060) by RGID:
rgid
1   -2448.0
2   -2655.0
3   -1067.0
4    -494.0
5   -4078.0
6   -2253.0
